In [1]:
import pandas as pd
import numpy as np
import ta
import datetime
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv( "C:/Users/DELL/Desktop/UOIT/capstone/AMZN_alpha_2min_2024-05_to_2025-05.csv")

 #Convert 'Timestamp' from object to datetime.
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

#check if it is  converted
print( "the data type of 'Timestap' in dataset is", df['Timestamp'].dtype)

the data type of 'Timestap' in dataset is datetime64[ns]


In [3]:
# Define regular trading hours

market_open = datetime.time(9, 30)
market_close = datetime.time(16, 0)

# Keep only rows within regular trading hours
df = df[
    (df['Timestamp'].dt.time >= market_open) & 
    (df['Timestamp'].dt.time <= market_close)
].copy()

print(f" First {df.head(1)}")
print(f" Last {df.tail(1)}")

 First               Timestamp     Open   High       Low   Close   Volume
165 2024-05-01 09:30:00  181.635  182.0  179.5907  179.65  2837082
 Last                  Timestamp    Open    High    Low   Close    Volume
119422 2025-04-30 16:00:00  184.32  188.81  183.7  188.11  20396609


In [7]:
print(df.columns)


Index(['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume'], dtype='object')


In [8]:
df.head(10)

,Timestamp,Open,High,Low,Close,Volume
165,2024-05-01 09:30:00,181.6350,182.000,179.5907,179.6500,2837082
166,2024-05-01 09:32:00,179.6603,179.680,178.2600,178.5300,1744718
167,2024-05-01 09:34:00,178.5700,180.210,178.4700,180.0600,1769687
168,2024-05-01 09:36:00,180.0550,180.400,179.3200,180.2100,1037180
169,2024-05-01 09:38:00,180.2000,180.785,179.9000,180.1150,1088382
170,2024-05-01 09:40:00,180.1150,180.770,179.7700,180.6750,979535
171,2024-05-01 09:42:00,180.6600,181.300,180.4400,180.9600,1309882
172,2024-05-01 09:44:00,180.9600,181.310,180.3500,181.1800,1062979
173,2024-05-01 09:46:00,181.1700,181.250,180.6700,180.9500,626783
174,2024-05-01 09:48:00,180.9500,181.500,180.9300,181.1399,983758


In [9]:
#---------------Feature ENgineering----------------#
#---------------Feature ENgineering----------------#
#---------------Feature ENgineering----------------#

In [10]:
# RCI_9
def rci(series, period=9):
    result = np.full(len(series), np.nan)
    for i in range(period-1, len(series)):
        price = series[i-period+1:i+1].to_numpy()
        time = np.arange(period, 0, -1)
        order = price.argsort().argsort() + 1  # rank
        diff = order - time
        rci_value = (1 - (6 * np.sum(diff**2)) / (period * (period**2 - 1))) * 100
        result[i] = rci_value
    return result

#  RCI (9 periods)
df['RCI_9'] = rci(df['Close'], period=14)

# CCI
df['CCI'] = ta.trend.cci(df['High'], df['Low'], df['Close'], window=20, fillna=True)


# 3. MACD
macd = ta.trend.macd(df['Close'], window_slow=26, window_fast=12)
macd_signal = ta.trend.macd_signal(df['Close'], window_slow=26, window_fast=12, window_sign=9)
df['MACD'] = macd
df['MACD_signal'] = macd_signal
df['MACD_hist'] = macd - macd_signal

# 4. KAMA
df['KAMA'] = ta.momentum.kama(df['Close'], window=10, pow1=2, pow2=30, fillna=True)

# 5. ROC
df['ROC'] = ta.momentum.roc(df['Close'], window=14, fillna=True)
# https://www.avatrade.ca/education/technical-analysis-indicators-strategies/roc-indicator-strategies#2

# 6. EMA12
df['EMA12'] = ta.trend.ema_indicator(df['Close'], window=12, fillna=True)
# https://www.thinkmarkets.com/en/trading-academy/forex/exponential-moving-averages/

# 7. Relative Strength Index (RSI)
df['RSI'] = ta.momentum.rsi(df['Close'], window=7, fillna=True)
# https://choiceindia.com/blog/best-rsi-setting-for-intraday

# 8. Money Flow Index (MFI)
df['MFI'] = ta.volume.money_flow_index(df['High'], df['Low'], df['Close'], df['Volume'], window=14, fillna=True)

# 9. Relative Vigor Index (RVI) -- not directly available, define custom
def rvi_strict(close, open_, high, low, window=10):
    # Calculate 4-bar weighted sum for numerator (Close-Open)
    a = close - open_
    b = close.shift(1) - open_.shift(1)
    c = close.shift(2) - open_.shift(2)
    d = close.shift(3) - open_.shift(3)
    numerator = (a + 2*b + 2*c + d) / 6

    # Calculate 4-bar weighted sum for denominator (High-Low)
    e = high - low
    f = high.shift(1) - low.shift(1)
    g = high.shift(2) - low.shift(2)
    h = high.shift(3) - low.shift(3)
    denominator = (e + 2*f + 2*g + h) / 6

    # Rolling mean for N periods
    num_sma = numerator.rolling(window=window).mean()
    denom_sma = denominator.rolling(window=window).mean()

    # RVI calculation
    rvi = num_sma / denom_sma
    return rvi


df['RVI'] = rvi_strict(df['Close'], df['Open'], df['High'], df['Low'], window=10)

# https://www.investopedia.com/terms/r/relative_vigor_index.asp
# https://www.avatrade.ca/education/technical-analysis-indicators-strategies/rvi-indicator-strategies

# 10. Williams %R
df['WilliamsR'] = ta.momentum.williams_r(df['High'], df['Low'], df['Close'], lbp=14, fillna=True)
# https://trendspider.com/learning-center/introduction-to-williams-r-range/

# 11. Percentage Price Oscillator (PPO)
df['PPO'] = ta.momentum.ppo(df['Close'], window_slow=26, window_fast=12, window_sign=9, fillna=True)
# https://www.investopedia.com/terms/p/ppo.asp

# 12. Pivot Point (Classic) -- daily
df['Pivot_Point'] = (df['High'] + df['Low'] + df['Close']) / 3

# 13. Stochastic Oscillator
# Calculate %K
df['Stoch_K'] = ta.momentum.stoch(df['High'], df['Low'], df['Close'], window=14, smooth_window=3, fillna=True)
# Calculate %D (the signal line) as SMA of %K
df['Stoch_D'] = df['Stoch_K'].rolling(window=3).mean()
# https://www.investopedia.com/terms/s/stochasticoscillator.asp

# 14. Internal Bar Strength (IBS)
df['IBS'] = (df['Close'] - df['Low']) / (df['High'] - df['Low'])

# 15. Chaikin Money Flow (CMF)
def chaikin_money_flow(df, window=20):
    mfv = ((df['Close'] - df['Low']) - (df['High'] - df['Close'])) / (df['High'] - df['Low']) * df['Volume']
    cmf = mfv.rolling(window=window).sum() / df['Volume'].rolling(window=window).sum()
    return cmf

df['CMF'] = chaikin_money_flow(df, window=20)
# https://help.stockdoctor.com.au/hc/en-us/articles/360000099635-Chaikin-Money-Flow

In [11]:
print(df.columns)
print("Number of columns:", len(df.columns))

Index(['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume', 'RCI_9', 'CCI',
       'MACD', 'MACD_signal', 'MACD_hist', 'KAMA', 'ROC', 'EMA12', 'RSI',
       'MFI', 'RVI', 'WilliamsR', 'PPO', 'Pivot_Point', 'Stoch_K', 'Stoch_D',
       'IBS', 'CMF'],
      dtype='object')
Number of columns: 24


In [12]:
# drop NaN from both datasets
df = df.dropna().reset_index(drop=True)

# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())

#Checking if duplicated rows exist.
duplicate_rows_df = df[df.duplicated()]
print("number of duplicate rows: ", duplicate_rows_df.shape)


Missing Values:
Timestamp      0
Open           0
High           0
Low            0
Close          0
Volume         0
RCI_9          0
CCI            0
MACD           0
MACD_signal    0
MACD_hist      0
KAMA           0
ROC            0
EMA12          0
RSI            0
MFI            0
RVI            0
WilliamsR      0
PPO            0
Pivot_Point    0
Stoch_K        0
Stoch_D        0
IBS            0
CMF            0
dtype: int64
number of duplicate rows:  (0, 24)


In [13]:
#--------------------------Evaluation------------------------#

In [30]:
# RCI max drawdown

rci_buy_std = -80   # Example buy threshold for RCI
rci_sell_std = 80   # Example sell threshold for RCI

holding_position = False      # Not in a trade to start
entry_index = None

buy_indices = []
sell_indices = []
each_return = []
trade_days = []  # Store number of days held per trade

# --- Main trading loop ---
for i in range(len(df) - 1):
    rci = df['RCI_9'].iloc[i]
    
    # ENTRY: Only if not already holding a position
    if not holding_position and rci < rci_buy_std:
        entry_index = i + 1  # Buy at NEXT bar's Open
        if entry_index >= len(df):  # Out-of-bounds check
            break
        holding_position = True

    # EXIT: Only if holding a position
    elif holding_position and rci > rci_sell_std:
        exit_index = i + 1  # Sell at NEXT bar's Open
        if exit_index >= len(df):  # Out-of-bounds check
            break
        
        # Debug
        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")
        
        # ---- Record trade info ----
        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        each_return.append(trade_return)
        
        # Add holding period in days
        trade_days.append(exit_index - entry_index)
        
        holding_position = False

# --- Performance metrics ---
sum_return = sum(each_return)
cumulative_return = (np.prod([1 + r for r in each_return]) - 1) if each_return else 0
average_return = sum(each_return) / len(each_return) if each_return else 0

# ---- Win rate ----
num_wins = sum(r > 0 for r in each_return)
winrate = num_wins / len(each_return) if each_return else 0

# ---- Max drawdown ----
if each_return:
    equity_curve = np.cumprod([1 + r for r in each_return])
    running_max = np.maximum.accumulate(equity_curve)
    drawdown = (equity_curve - running_max) / running_max
    max_drawdown = drawdown.min()
else:
    max_drawdown = 0

# ---- Average return per day (only for held trade days) ----
total_days = sum(trade_days) if trade_days else 0
average_return_per_day = (sum(each_return) / total_days) if total_days > 0 else 0

print("RCI")
print(f"Number of trades: {len(each_return)}")
print(f"Sum of return: {sum_return:.4f}")
print(f"Cumulative return: {cumulative_return:.4%}")
print(f"Average return per trade: {average_return:.4}")
print(f"Average return per day (while in trade): {average_return_per_day:.10f}")
print(f"Win rate: {winrate:.2%}")
print(f"Max drawdown: {max_drawdown:.2%}")

print("\nSample trades:")
for i in range(min(10, len(each_return))):
    print(f"Trade {i+1}: Entry at {buy_indices[i]}, Exit at {sell_indices[i]}, "
          f"Days held: {trade_days[i]}, Each Return: {each_return[i]:.4%}")



RCI
Number of trades: 627
Sum of return: 0.1427
Cumulative return: 11.1701%
Average return per trade: 0.0002275
Average return per day (while in trade): 0.0000056288
Win rate: 38.60%
Max drawdown: -25.28%

Sample trades:
Trade 1: Entry at 1, Exit at 13, Days held: 12, Each Return: -1.0445%
Trade 2: Entry at 34, Exit at 43, Days held: 9, Each Return: -1.1750%
Trade 3: Entry at 67, Exit at 95, Days held: 28, Each Return: -0.1523%
Trade 4: Entry at 104, Exit at 154, Days held: 50, Each Return: 1.2079%
Trade 5: Entry at 171, Exit at 188, Days held: 17, Each Return: -1.1131%
Trade 6: Entry at 211, Exit at 225, Days held: 14, Each Return: -0.3706%
Trade 7: Entry at 236, Exit at 312, Days held: 76, Each Return: 1.1717%
Trade 8: Entry at 320, Exit at 408, Days held: 88, Each Return: 0.9304%
Trade 9: Entry at 427, Exit at 443, Days held: 16, Each Return: -0.0804%
Trade 10: Entry at 473, Exit at 481, Days held: 8, Each Return: -0.1957%


In [ ]:
# CCI standard threshold

cci_buy_std = -100   # Common buy threshold for CCI
cci_sell_std = 100   # Common sell threshold for CCI

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
returns = []

for i in range(len(df) - 1):
    cci = df['CCI'].iloc[i]

    # ENTRY: Only if not already holding a position
    if not holding_position and cci < cci_buy_std:
        entry_index = i + 1  # Buy at NEXT bar's Open
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: Only if holding a position
    elif holding_position and cci > cci_sell_std:
        exit_index = i + 1  # Sell at NEXT bar's Open
        if exit_index >= len(df):
            break

        # Debug: check for same-bar trade
        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        returns.append(trade_return)

        holding_position = False


sum_return = sum(returns)
cumulative_return = (np.prod([1 + r for r in returns]) - 1) if returns else 0
average_return = sum(returns) / len(returns) if each_return else 0

print("CCI")
print(f"Number of trades: {len(returns)}")
print(f"Sum of return: {sum_return:.4f}")
print(f"Cumulative return: {cumulative_return:.4%}\n")

print(f"Average return per trade: {average_return:.4}")

print("Sample trades:")
for i in range(min(10, len(returns))):
    print(f"Trade {i+1}: Entry at {buy_indices[i]}, Exit at {sell_indices[i]}, Each Return: {returns[i]:.4%}")

print(f"\nAverage return per trade: {np.mean(returns):.6f}")
median_return = np.median(returns) if returns else 0

print(f"\nMedian return per trade: {median_return:.6f}")


In [17]:
# --- MACD Crossover Rule-Based Backtest (Go Long Only) ---

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
returns = []

for i in range(1, len(df) - 1):  # Start from 1 to use previous bar for crossover check
    macd_prev = df['MACD'].iloc[i-1] #previous macd 
    macd_sig_prev = df['MACD_signal'].iloc[i-1]
    macd = df['MACD'].iloc[i]
    macd_sig = df['MACD_signal'].iloc[i]
    
    # ENTRY: MACD crosses above MACD_signal (bullish crossover)
    # ENTRY: Bullish crossover AND both below zero
    if (
        not holding_position
        and (macd_prev <= macd_sig_prev) and (macd > macd_sig)
        and (macd < 0) and (macd_sig < 0)
    ):
        entry_index = i + 1
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: MACD crosses below MACD_signal (bearish crossover)
    elif holding_position and (macd_prev >= macd_sig_prev) and (macd < macd_sig):
        exit_index = i + 1  # Sell at NEXT bar's Open
        if exit_index >= len(df):
            break

        # Debug: check for same-bar trade
        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        returns.append(trade_return)

        holding_position = False


sum_return = sum(returns)
cumulative_return = (np.prod([1 + r for r in returns]) - 1) if returns else 0

print("MACD Crossover")
print(f"Number of trades: {len(returns)}")
print(f"Sum of return: {sum_return:.4f}")
print(f"Cumulative return: {cumulative_return:.4%}\n")
average_return = sum(returns) / len(returns) if each_return else 0
print(f"Average return per trade: {average_return:.4}")

print("Sample trades:")
for i in range(min(10, len(returns))):
    print(f"Trade {i+1}: Entry at {buy_indices[i]}, Exit at {sell_indices[i]}, Each Return: {returns[i]:.4%}")

print(f"\nAverage return per trade: {np.mean(returns):.6f}")



MACD Crossover
Number of trades: 1166
Sum of return: 0.1547
Cumulative return: 14.0918%

Average return per trade: 0.0001327
Sample trades:
Trade 1: Entry at 28, Exit at 41, Each Return: -0.3199%
Trade 2: Entry at 50, Exit at 89, Each Return: 0.4512%
Trade 3: Entry at 166, Exit at 183, Each Return: 0.2567%
Trade 4: Entry at 195, Exit at 201, Each Return: -0.4586%
Trade 5: Entry at 203, Exit at 220, Each Return: 0.0220%
Trade 6: Entry at 234, Exit at 246, Each Return: 0.0206%
Trade 7: Entry at 356, Exit at 374, Each Return: 0.9804%
Trade 8: Entry at 422, Exit at 441, Each Return: 0.0805%
Trade 9: Entry at 468, Exit at 479, Each Return: -0.1233%
Trade 10: Entry at 490, Exit at 509, Each Return: 0.0698%

Average return per trade: 0.000133


In [31]:
# MACD, winrate

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
returns = []
trade_days = []

for i in range(1, len(df) - 1):  # Start from 1 to use previous bar for crossover check
    macd_prev = df['MACD'].iloc[i-1]
    macd_sig_prev = df['MACD_signal'].iloc[i-1]
    macd = df['MACD'].iloc[i]
    macd_sig = df['MACD_signal'].iloc[i]
    
    # ENTRY: Bullish crossover AND both below zero
    if (
        not holding_position
        and (macd_prev <= macd_sig_prev) and (macd > macd_sig)
        and (macd < 0) and (macd_sig < 0)
    ):
        entry_index = i + 1
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: Bearish crossover
    elif holding_position and (macd_prev >= macd_sig_prev) and (macd < macd_sig):
        exit_index = i + 1  # Sell at NEXT bar's Open
        if exit_index >= len(df):
            break

        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        returns.append(trade_return)
        trade_days.append(exit_index - entry_index)

        holding_position = False

sum_return = sum(returns)
cumulative_return = (np.prod([1 + r for r in returns]) - 1) if returns else 0
average_return = sum(returns) / len(returns) if returns else 0

# Win rate
num_wins = sum(r > 0 for r in returns)
winrate = num_wins / len(returns) if returns else 0

# Max drawdown
if returns:
    equity_curve = np.cumprod([1 + r for r in returns])
    running_max = np.maximum.accumulate(equity_curve)
    drawdown = (equity_curve - running_max) / running_max
    max_drawdown = drawdown.min()
else:
    max_drawdown = 0

# Average return per day (while in trade)
total_days = sum(trade_days) if trade_days else 0
average_return_per_day = (sum(returns) / total_days) if total_days > 0 else 0

print("MACD Crossover")
print(f"Number of trades: {len(returns)}")
print(f"Sum of return: {sum_return:.4f}")
print(f"Cumulative return: {cumulative_return:.4%}")
print(f"Average return per trade: {average_return:.4f}")
print(f"Average return per day (while in trade): {average_return_per_day:.10f}")
print(f"Win rate: {winrate:.2%}")
print(f"Max drawdown: {max_drawdown:.2%}")

print("\nSample trades:")
for i in range(min(10, len(returns))):
    print(f"Trade {i+1}: Entry at {buy_indices[i]}, Exit at {sell_indices[i]}, Days held: {trade_days[i]}, Each Return: {returns[i]:.4%}")



MACD Crossover
Number of trades: 1166
Sum of return: 0.1547
Cumulative return: 14.0918%
Average return per trade: 0.0001
Average return per day (while in trade): 0.0000083986
Win rate: 36.62%
Max drawdown: -19.93%

Sample trades:
Trade 1: Entry at 28, Exit at 41, Days held: 13, Each Return: -0.3199%
Trade 2: Entry at 50, Exit at 89, Days held: 39, Each Return: 0.4512%
Trade 3: Entry at 166, Exit at 183, Days held: 17, Each Return: 0.2567%
Trade 4: Entry at 195, Exit at 201, Days held: 6, Each Return: -0.4586%
Trade 5: Entry at 203, Exit at 220, Days held: 17, Each Return: 0.0220%
Trade 6: Entry at 234, Exit at 246, Days held: 12, Each Return: 0.0206%
Trade 7: Entry at 356, Exit at 374, Days held: 18, Each Return: 0.9804%
Trade 8: Entry at 422, Exit at 441, Days held: 19, Each Return: 0.0805%
Trade 9: Entry at 468, Exit at 479, Days held: 11, Each Return: -0.1233%
Trade 10: Entry at 490, Exit at 509, Days held: 19, Each Return: 0.0698%


In [ ]:
# KAMA
df['EMA200'] = df['Close'].ewm(span=200, adjust=False).mean() #use EMA200 as the reference

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
returns = []

for i in range(1, len(df) - 1):
    kama_prev = df['KAMA'].iloc[i-1]
    ema200_prev = df['EMA200'].iloc[i-1]
    kama = df['KAMA'].iloc[i]
    ema200 = df['EMA200'].iloc[i]
    
    # ENTRY: KAMA crosses above EMA200
    if (not holding_position) and (kama_prev <= ema200_prev) and (kama > ema200):
        entry_index = i + 1
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: KAMA crosses below EMA200
    elif holding_position and (kama_prev >= ema200_prev) and (kama < ema200):
        exit_index = i + 1
        if exit_index >= len(df):
            break

        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        returns.append(trade_return)

        holding_position = False

sum_return = sum(returns)
cumulative_return = (np.prod([1 + r for r in returns]) - 1) if returns else 0

print("MACD Crossover")
print(f"Number of trades: {len(returns)}")
print(f"Sum of return: {sum_return:.4f}")
print(f"Cumulative return: {cumulative_return:.4%}\n")
average_return = sum(each_return) / len(each_return) if each_return else 0
print(f"Average return per trade: {average_return:.4}")

print("Sample trades:")
for i in range(min(10, len(returns))):
    print(f"Trade {i+1}: Entry at {buy_indices[i]}, Exit at {sell_indices[i]}, Each Return: {returns[i]:.4%}")

print(f"\nAverage return per trade: {np.mean(returns):.6f}")


In [ ]:
# EMA12
holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
returns = []

for i in range(1, len(df) - 1):
    close_prev = df['Close'].iloc[i-1]
    ema12_prev = df['EMA12'].iloc[i-1]
    close = df['Close'].iloc[i]
    ema12 = df['EMA12'].iloc[i]
    
    # ENTRY: Close crosses above EMA12
    if (not holding_position) and (close_prev <= ema12_prev) and (close > ema12):
        entry_index = i + 1
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: Close crosses below EMA12
    elif holding_position and (close_prev >= ema12_prev) and (close < ema12):
        exit_index = i + 1
        if exit_index >= len(df):
            break

        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        returns.append(trade_return)

        holding_position = False


sum_return = sum(returns)
cumulative_return = (np.prod([1 + r for r in returns]) - 1) if returns else 0

print("EMA12 Crossover")
print(f"Number of trades: {len(returns)}")
print(f"Sum of return: {sum_return:.4f}")
print(f"Cumulative return: {cumulative_return:.4%}\n")
average_return = sum(each_return) / len(each_return) if each_return else 0
print(f"Average return per trade: {average_return:.4}")

print("Sample trades:")
for i in range(min(10, len(returns))):
    print(f"Trade {i+1}: Entry at {buy_indices[i]}, Exit at {sell_indices[i]}, Each Return: {returns[i]:.4%}")

print(f"\nAverage return per trade: {np.mean(returns):.6f}")


In [33]:
# ROC

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
returns = []
trade_days = []

for i in range(1, len(df) - 1):
    roc_prev = df['ROC'].iloc[i-1]
    roc = df['ROC'].iloc[i]

    # ENTRY: ROC crosses above 0
    if (not holding_position) and (roc_prev <= 0) and (roc > 0):
        entry_index = i + 1
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: ROC crosses below 0
    elif holding_position and (roc_prev >= 0) and (roc < 0):
        exit_index = i + 1
        if exit_index >= len(df):
            break

        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        returns.append(trade_return)
        trade_days.append(exit_index - entry_index)

        holding_position = False

# --- Results Summary and Sample Trades ---
sum_return = sum(returns)
cumulative_return = (np.prod([1 + r for r in returns]) - 1) if returns else 0
average_return = sum(returns) / len(returns) if returns else 0

# Win rate
num_wins = sum(r > 0 for r in returns)
winrate = num_wins / len(returns) if returns else 0

# Max drawdown
if returns:
    equity_curve = np.cumprod([1 + r for r in returns])
    running_max = np.maximum.accumulate(equity_curve)
    drawdown = (equity_curve - running_max) / running_max
    max_drawdown = drawdown.min()
else:
    max_drawdown = 0

# Average return per day (while in trade)
total_days = sum(trade_days) if trade_days else 0
average_return_per_day = (sum(returns) / total_days) if total_days > 0 else 0

print("ROC Zero Line Crossover")
print(f"Number of trades: {len(returns)}")
print(f"Sum of return: {sum_return:.4f}")
print(f"Cumulative return: {cumulative_return:.4%}")
print(f"Average return per trade: {average_return:.4f}")
print(f"Average return per day (while in trade): {average_return_per_day:.10f}")
print(f"Win rate: {winrate:.2%}")
print(f"Max drawdown: {max_drawdown:.2%}")

print("\nSample trades:")
for i in range(min(10, len(returns))):
    print(f"Trade {i+1}: Entry at {buy_indices[i]}, Exit at {sell_indices[i]}, Days held: {trade_days[i]}, Each Return: {returns[i]:.4%}")




ROC Zero Line Crossover
Number of trades: 2960
Sum of return: 0.0737
Cumulative return: 4.2496%
Average return per trade: 0.0000
Average return per day (while in trade): 0.0000029685
Win rate: 33.82%
Max drawdown: -23.58%

Sample trades:
Trade 1: Entry at 11, Exit at 12, Days held: 1, Each Return: -0.5693%
Trade 2: Entry at 31, Exit at 41, Days held: 10, Each Return: -0.5940%
Trade 3: Entry at 57, Exit at 79, Days held: 22, Each Return: 0.1746%
Trade 4: Entry at 80, Exit at 81, Days held: 1, Each Return: -0.0112%
Trade 5: Entry at 82, Exit at 90, Days held: 8, Each Return: -0.1543%
Trade 6: Entry at 100, Exit at 147, Days held: 47, Each Return: 3.3279%
Trade 7: Entry at 168, Exit at 183, Days held: 15, Each Return: -0.2677%
Trade 8: Entry at 196, Exit at 200, Days held: 4, Each Return: -0.3926%
Trade 9: Entry at 201, Exit at 209, Days held: 8, Each Return: 0.4028%
Trade 10: Entry at 210, Exit at 222, Days held: 12, Each Return: -0.4059%


In [ ]:
# RSI
rsi_buy_std = 30   # Example buy threshold for RSI (oversold)
rsi_sell_std = 70  # Example sell threshold for RSI (overbought)

holding_position = False      # Not in a trade to start
entry_index = None

buy_indices = []
sell_indices = []
each_return = []

# --- Main trading loop ---
for i in range(len(df) - 1):
    rsi = df['RSI'].iloc[i]
    
    # ENTRY: Only if not already holding a position
    if not holding_position and rsi < rsi_buy_std:
        entry_index = i + 1  # Buy at NEXT bar's Open
        if entry_index >= len(df):  # Out-of-bounds check
            break
        holding_position = True

    # EXIT: Only if holding a position
    elif holding_position and rsi > rsi_sell_std:
        exit_index = i + 1  # Sell at NEXT bar's Open
        if exit_index >= len(df):  # Out-of-bounds check
            break
        
        # Debug
        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")
        
        # ---- Record trade info ----
        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        each_return.append(trade_return)
        
        holding_position = False

# 
sum_return = sum(each_return)
cumulative_return = (np.prod([1 + r for r in each_return]) - 1) if each_return else 0
average_return = sum(each_return) / len(each_return) if each_return else 0

print("RSI")
print(f"Number of trades: {len(each_return)}")
print(f"Sum of return: {sum_return:.4f}")
print(f"Cumulative return: {cumulative_return:.4%}")
print(f"Average return per trade: {average_return:.4}")

print("\nSample trades:")
for i in range(min(10, len(each_return))):
    print(f"Trade {i+1}: Entry at {buy_indices[i]}, Exit at {sell_indices[i]}, Each Return: {each_return[i]:.4%}")

print(f"Average return per trade: {np.mean(each_return):.6f}")


In [ ]:
# MFI
mfi_buy_std = 20   # Buy threshold for MFI (oversold)
mfi_sell_std = 80  # Sell threshold for MFI (overbought)

holding_position = False      # Not in a trade to start
entry_index = None

buy_indices = []
sell_indices = []
each_return = []

# --- Main trading loop ---
for i in range(len(df) - 1):
    mfi = df['MFI'].iloc[i]
    
    # ENTRY: Only if not already holding a position
    if not holding_position and mfi < mfi_buy_std:
        entry_index = i + 1  # Buy at NEXT bar's Open
        if entry_index >= len(df):  # Out-of-bounds check
            break
        holding_position = True

    # EXIT: Only if holding a position
    elif holding_position and mfi > mfi_sell_std:
        exit_index = i + 1  # Sell at NEXT bar's Open
        if exit_index >= len(df):  # Out-of-bounds check
            break
        
        # Debug
        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")
        
        # ---- Record trade info ----
        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        each_return.append(trade_return)
        
        holding_position = False

# 
sum_return = sum(each_return)
cumulative_return = (np.prod([1 + r for r in each_return]) - 1) if each_return else 0
average_return = sum(each_return) / len(each_return) if each_return else 0

print("MFI")
print(f"Number of trades: {len(each_return)}")
print(f"Sum of return: {sum_return:.4f}")
print(f"Cumulative return: {cumulative_return:.4%}")
print(f"Average return per trade: {average_return:.4}")

print("\nSample trades:")
for i in range(min(10, len(each_return))):
    print(f"Trade {i+1}: Entry at {buy_indices[i]}, Exit at {sell_indices[i]}, Each Return: {each_return[i]:.4%}")

print(f"Average return per trade: {np.mean(each_return):.6f}")


In [ ]:
# RVI (using RSI crossover)
holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
each_return = []

# --- Main trading loop ---
for i in range(1, len(df) - 1):
    rvi_prev = df['RVI'].iloc[i - 1]
    rsi_prev = df['RSI'].iloc[i - 1]
    rvi = df['RVI'].iloc[i]
    rsi = df['RSI'].iloc[i]
    
    # ENTRY: RVI crosses above RSI
    if not holding_position and (rvi_prev < rsi_prev) and (rvi > rsi):
        entry_index = i + 1  # Buy at NEXT bar's Open
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: RVI crosses below RSI
    elif holding_position and (rvi_prev > rsi_prev) and (rvi < rsi):
        exit_index = i + 1  # Sell at NEXT bar's Open
        if exit_index >= len(df):
            break

        # Debug
        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        # ---- Record trade info ----
        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        each_return.append(trade_return)

        holding_position = False

# 
sum_return = sum(each_return)
cumulative_return = (np.prod([1 + r for r in each_return]) - 1) if each_return else 0
average_return = sum(each_return) / len(each_return) if each_return else 0

print("RVI/RSI Crossover Strategy")
print(f"Number of trades: {len(each_return)}")
print(f"Sum of return: {sum_return:.4f}")
print(f"Cumulative return: {cumulative_return:.4%}")
print(f"Average return per trade: {average_return:.4f}")

print("\nSample trades:")
for i in range(min(10, len(each_return))):
    print(f"Trade {i+1}: Entry at {buy_indices[i]}, Exit at {sell_indices[i]}, Each Return: {each_return[i]:.4%}")

print(f"Average return per trade: {np.mean(each_return):.6f}")


In [ ]:
print("RVI/RSI Crossover Strategy")
print(f"Number of trades: {len(each_return)}")
print(f"Sum of return: {sum_return:.4f}")
print(f"Cumulative return: {cumulative_return:.4%}")
if len(each_return) > 0:
    print(f"Average return per trade: {average_return:.4f}")
else:
    print("Average return per trade: N/A (no trades executed)")

print("\nSample trades:")
if len(each_return) > 0:
    for i in range(min(10, len(each_return))):
        print(f"Trade {i+1}: Entry at {buy_indices[i]}, Exit at {sell_indices[i]}, Each Return: {each_return[i]:.4%}")
    print(f"Average return per trade: {np.mean(each_return):.6f}")
else:
    print("No sample trades: No trades executed.")


In [ ]:
# William
wpr_buy_std = -80    # Buy threshold for Williams %R (oversold)
wpr_sell_std = -20   # Sell threshold for Williams %R (overbought)

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
each_return = []

# --- Main trading loop ---
for i in range(len(df) - 1):
    wpr = df['WilliamsR'].iloc[i]   # Replace with your actual column name if different

    # ENTRY: Only if not already holding a position
    if not holding_position and wpr < wpr_buy_std:
        entry_index = i + 1  # Buy at NEXT bar's Open
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: Only if holding a position
    elif holding_position and wpr > wpr_sell_std:
        exit_index = i + 1  # Sell at NEXT bar's Open
        if exit_index >= len(df):
            break

        # Debug
        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        # ---- Record trade info ----
        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        each_return.append(trade_return)

        holding_position = False

#
sum_return = sum(each_return)
cumulative_return = (np.prod([1 + r for r in each_return]) - 1) if each_return else 0
average_return = sum(each_return) / len(each_return) if each_return else 0

print("Williams %R Strategy")
print(f"Number of trades: {len(each_return)}")
print(f"Sum of return: {sum_return:.4f}")
print(f"Cumulative return: {cumulative_return:.4%}")
if len(each_return) > 0:
    print(f"Average return per trade: {average_return:.4f}")
else:
    print("Average return per trade: N/A (no trades executed)")

print("\nSample trades:")
if len(each_return) > 0:
    for i in range(min(10, len(each_return))):
        print(f"Trade {i+1}: Entry at {buy_indices[i]}, Exit at {sell_indices[i]}, Each Return: {each_return[i]:.4%}")
else:
    print("No sample trades: No trades executed.")


In [ ]:
# PPO
ppo_buy_std = -10    # Buy threshold for PPO (oversold)
ppo_sell_std = 10    # Sell threshold for PPO (overbought)

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
each_return = []

# --- Main trading loop ---
for i in range(len(df) - 1):
    ppo = df['PPO'].iloc[i]   # Replace with your actual PPO column name if different

    # ENTRY: Only if not already holding a position
    if not holding_position and ppo < ppo_buy_std:
        entry_index = i + 1  # Buy at NEXT bar's Open
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: Only if holding a position
    elif holding_position and ppo > ppo_sell_std:
        exit_index = i + 1  # Sell at NEXT bar's Open
        if exit_index >= len(df):
            break

        # Debug
        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        # ---- Record trade info ----
        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        each_return.append(trade_return)

        holding_position = False

#
sum_return = sum(each_return)
cumulative_return = (np.prod([1 + r for r in each_return]) - 1) if each_return else 0
average_return = sum(each_return) / len(each_return) if each_return else 0

print("PPO Strategy")
print(f"Number of trades: {len(each_return)}")
print(f"Sum of return: {sum_return:.4f}")
print(f"Cumulative return: {cumulative_return:.4%}")
if len(each_return) > 0:
    print(f"Average return per trade: {average_return:.4f}")
else:
    print("Average return per trade: N/A (no trades executed)")

print("\nSample trades:")
if len(each_return) > 0:
    for i in range(min(10, len(each_return))):
        print(f"Trade {i+1}: Entry at {buy_indices[i]}, Exit at {sell_indices[i]}, Each Return: {each_return[i]:.4%}")
else:
    print("No sample trades: No trades executed.")


In [ ]:
# Stocj L
stoch_buy_std = 20    # Buy threshold for Stochastic Oscillator (oversold)
stoch_sell_std = 80   # Sell threshold for Stochastic Oscillator (overbought)

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
each_return = []

# --- Main trading loop ---
for i in range(len(df) - 1):
    stoch = df['Stoch_K'].iloc[i]   # Replace with your actual column name if needed

    # ENTRY: Only if not already holding a position
    if not holding_position and stoch < stoch_buy_std:
        entry_index = i + 1  # Buy at NEXT bar's Open
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: Only if holding a position
    elif holding_position and stoch > stoch_sell_std:
        exit_index = i + 1  # Sell at NEXT bar's Open
        if exit_index >= len(df):
            break

        # Debug
        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        # ---- Record trade info ----
        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        each_return.append(trade_return)

        holding_position = False

#
sum_return = sum(each_return)
cumulative_return = (np.prod([1 + r for r in each_return]) - 1) if each_return else 0
average_return = sum(each_return) / len(each_return) if each_return else 0

print("Stochastic Oscillator Strategy")
print(f"Number of trades: {len(each_return)}")
print(f"Sum of return: {sum_return:.4f}")
print(f"Cumulative return: {cumulative_return:.4%}")
if len(each_return) > 0:
    print(f"Average return per trade: {average_return:.4f}")
else:
    print("Average return per trade: N/A (no trades executed)")

print("\nSample trades:")
if len(each_return) > 0:
    for i in range(min(10, len(each_return))):
        print(f"Trade {i+1}: Entry at {buy_indices[i]}, Exit at {sell_indices[i]}, Each Return: {each_return[i]:.4%}")
else:
    print("No sample trades: No trades executed.")


In [ ]:
print(df[['Stoch_K', 'WilliamsR']].head(20))


In [29]:
# IBS

ibs_buy_std = 0.2   # Buy threshold for IBS (oversold)
ibs_sell_std = 0.9  # Sell threshold for IBS (overbought)

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
each_return = []
trade_days = []

# --- Main trading loop ---
for i in range(len(df) - 1):
    ibs = df['IBS'].iloc[i]   # Replace with your IBS column name if different

    # ENTRY: Only if not already holding a position
    if not holding_position and ibs < ibs_buy_std:
        entry_index = i + 1  # Buy at NEXT bar's Open
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: Only if holding a position
    elif holding_position and ibs > ibs_sell_std:
        exit_index = i + 1  # Sell at NEXT bar's Open
        if exit_index >= len(df):
            break

        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        # ---- Record trade info ----
        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        each_return.append(trade_return)
        trade_days.append(exit_index - entry_index)

        holding_position = False

# --- Performance metrics ---
sum_return = sum(each_return)
cumulative_return = (np.prod([1 + r for r in each_return]) - 1) if each_return else 0
average_return = sum(each_return) / len(each_return) if each_return else 0

# Win rate
num_wins = sum(r > 0 for r in each_return)
winrate = num_wins / len(each_return) if each_return else 0

# Max drawdown
if each_return:
    equity_curve = np.cumprod([1 + r for r in each_return])
    running_max = np.maximum.accumulate(equity_curve)
    drawdown = (equity_curve - running_max) / running_max
    max_drawdown = drawdown.min()
else:
    max_drawdown = 0

# Average return per day (while in trade)
total_days = sum(trade_days) if trade_days else 0
average_return_per_day = (sum(each_return) / total_days) if total_days > 0 else 0

print("IBS Strategy")
print(f"Number of trades: {len(each_return)}")
print(f"Sum of return: {sum_return:.4f}")
print(f"Cumulative return: {cumulative_return:.4%}")
print(f"Average return per trade: {average_return:.4f}" if len(each_return) > 0 else "Average return per trade: N/A (no trades executed)")
print(f"Average return per day (while in trade): {average_return_per_day:.10f}")
print(f"Win rate: {winrate:.2%}")
print(f"Max drawdown: {max_drawdown:.2%}")

print("\nSample trades:")
if len(each_return) > 0:
    for i in range(min(10, len(each_return))):
        print(f"Trade {i+1}: Entry at {buy_indices[i]}, Exit at {sell_indices[i]}, Days held: {trade_days[i]}, Each Return: {each_return[i]:.4%}")
else:
    print("No sample trades: No trades executed.")




IBS Strategy
Number of trades: 4143
Sum of return: 0.1319
Cumulative return: 9.0141%
Average return per trade: 0.0000
Average return per day (while in trade): 0.0000043970
Win rate: 61.12%
Max drawdown: -28.09%

Sample trades:
Trade 1: Entry at 5, Exit at 19, Days held: 14, Each Return: -1.2311%
Trade 2: Entry at 24, Exit at 26, Days held: 2, Each Return: 0.0325%
Trade 3: Entry at 35, Exit at 49, Days held: 14, Each Return: -0.7253%
Trade 4: Entry at 52, Exit at 61, Days held: 9, Each Return: 0.1070%
Trade 5: Entry at 74, Exit at 76, Days held: 2, Each Return: 0.0872%
Trade 6: Entry at 79, Exit at 80, Days held: 1, Each Return: 0.0871%
Trade 7: Entry at 85, Exit at 93, Days held: 8, Each Return: -0.4478%
Trade 8: Entry at 94, Exit at 96, Days held: 2, Each Return: 0.1461%
Trade 9: Entry at 110, Exit at 111, Days held: 1, Each Return: 0.0391%
Trade 10: Entry at 125, Exit at 128, Days held: 3, Each Return: 0.0164%


In [25]:
# CMF
cmf_buy_std = -0.25   # More conservative: use -0.20 if you prefer
cmf_sell_std = 0.25   # More conservative: use 0.20 if you prefer

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
each_return = []

# --- Main trading loop ---
for i in range(len(df) - 1):
    cmf = df['CMF'].iloc[i]   # Replace with your CMF column name if different

    # ENTRY: Only if not already holding a position
    if not holding_position and cmf < cmf_buy_std:
        entry_index = i + 1  # Buy at NEXT bar's Open
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: Only if holding a position
    elif holding_position and cmf > cmf_sell_std:
        exit_index = i + 1  # Sell at NEXT bar's Open
        if exit_index >= len(df):
            break

        # Debug
        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        # ---- Record trade info ----
        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        each_return.append(trade_return)

        holding_position = False

#
sum_return = sum(each_return)
cumulative_return = (np.prod([1 + r for r in each_return]) - 1) if each_return else 0
average_return = sum(each_return) / len(each_return) if each_return else 0

print("CMF Strategy")
print(f"Number of trades: {len(each_return)}")
print(f"Sum of return: {sum_return:.4f}")
print(f"Cumulative return: {cumulative_return:.4%}")
if len(each_return) > 0:
    print(f"Average return per trade: {average_return:.4f}")
else:
    print("Average return per trade: N/A (no trades executed)")

print("\nSample trades:")
if len(each_return) > 0:
    for i in range(min(10, len(each_return))):
        print(f"Trade {i+1}: Entry at {buy_indices[i]}, Exit at {sell_indices[i]}, Each Return: {each_return[i]:.4%}")
else:
    print("No sample trades: No trades executed.")


CMF Strategy
Number of trades: 240
Sum of return: -0.0900
Cumulative return: -10.7825%
Average return per trade: -0.0004

Sample trades:
Trade 1: Entry at 770, Exit at 800, Each Return: 0.1112%
Trade 2: Entry at 899, Exit at 943, Each Return: 0.4478%
Trade 3: Entry at 965, Exit at 1089, Each Return: 0.4842%
Trade 4: Entry at 1143, Exit at 1177, Each Return: 0.3388%
Trade 5: Entry at 1323, Exit at 1389, Each Return: -1.2772%
Trade 6: Entry at 1529, Exit at 1613, Each Return: -0.6934%
Trade 7: Entry at 1633, Exit at 1665, Each Return: 0.1237%
Trade 8: Entry at 1926, Exit at 1959, Each Return: -1.8710%
Trade 9: Entry at 2005, Exit at 2063, Each Return: -0.0485%
Trade 10: Entry at 2264, Exit at 2425, Each Return: -0.0885%


In [28]:
# CMF

cmf_buy_std = -0.25   # Buy threshold for CMF (more conservative: use -0.20 if you prefer)
cmf_sell_std = 0.25   # Sell threshold for CMF (more conservative: use 0.20 if you prefer)

holding_position = False
entry_index = None

buy_indices = []
sell_indices = []
each_return = []
trade_days = []

# --- Main trading loop ---
for i in range(len(df) - 1):
    cmf = df['CMF'].iloc[i]   # Replace with your CMF column name if different

    # ENTRY: Only if not already holding a position
    if not holding_position and cmf < cmf_buy_std:
        entry_index = i + 1  # Buy at NEXT bar's Open
        if entry_index >= len(df):
            break
        holding_position = True

    # EXIT: Only if holding a position
    elif holding_position and cmf > cmf_sell_std:
        exit_index = i + 1  # Sell at NEXT bar's Open
        if exit_index >= len(df):
            break

        if entry_index == exit_index:
            print(f"Warning: Buy and Sell at the same bar {entry_index}")

        # ---- Record trade info ----
        buy_indices.append(entry_index)
        sell_indices.append(exit_index)
        buy_price = df['Open'].iloc[entry_index]
        sell_price = df['Open'].iloc[exit_index]
        trade_return = (sell_price - buy_price) / buy_price
        each_return.append(trade_return)
        trade_days.append(exit_index - entry_index)

        holding_position = False

# --- Performance metrics ---
sum_return = sum(each_return)
cumulative_return = (np.prod([1 + r for r in each_return]) - 1) if each_return else 0
average_return = sum(each_return) / len(each_return) if each_return else 0

# Win rate
num_wins = sum(r > 0 for r in each_return)
winrate = num_wins / len(each_return) if each_return else 0

# Max drawdown
if each_return:
    equity_curve = np.cumprod([1 + r for r in each_return])
    running_max = np.maximum.accumulate(equity_curve)
    drawdown = (equity_curve - running_max) / running_max
    max_drawdown = drawdown.min()
else:
    max_drawdown = 0

# Average return per day (while in trade)
total_days = sum(trade_days) if trade_days else 0
average_return_per_day = (sum(each_return) / total_days) if total_days > 0 else 0

print("CMF Strategy")
print(f"Number of trades: {len(each_return)}")
print(f"Sum of return: {sum_return:.4f}")
print(f"Cumulative return: {cumulative_return:.4%}")
print(f"Average return per trade: {average_return:.4f}" if len(each_return) > 0 else "Average return per trade: N/A (no trades executed)")
print(f"Average return per day (while in trade): {average_return_per_day:.10f}")
print(f"Win rate: {winrate:.2%}")
print(f"Max drawdown: {max_drawdown:.2%}")

print("\nSample trades:")
if len(each_return) > 0:
    for i in range(min(10, len(each_return))):
        print(f"Trade {i+1}: Entry at {buy_indices[i]}, Exit at {sell_indices[i]}, Days held: {trade_days[i]}, Each Return: {each_return[i]:.4%}")
else:
    print("No sample trades: No trades executed.")



CMF Strategy
Number of trades: 240
Sum of return: -0.0900
Cumulative return: -10.7825%
Average return per trade: -0.0004
Average return per day (while in trade): -0.0000044649
Win rate: 55.83%
Max drawdown: -24.58%

Sample trades:
Trade 1: Entry at 770, Exit at 800, Days held: 30, Each Return: 0.1112%
Trade 2: Entry at 899, Exit at 943, Days held: 44, Each Return: 0.4478%
Trade 3: Entry at 965, Exit at 1089, Days held: 124, Each Return: 0.4842%
Trade 4: Entry at 1143, Exit at 1177, Days held: 34, Each Return: 0.3388%
Trade 5: Entry at 1323, Exit at 1389, Days held: 66, Each Return: -1.2772%
Trade 6: Entry at 1529, Exit at 1613, Days held: 84, Each Return: -0.6934%
Trade 7: Entry at 1633, Exit at 1665, Days held: 32, Each Return: 0.1237%
Trade 8: Entry at 1926, Exit at 1959, Days held: 33, Each Return: -1.8710%
Trade 9: Entry at 2005, Exit at 2063, Days held: 58, Each Return: -0.0485%
Trade 10: Entry at 2264, Exit at 2425, Days held: 161, Each Return: -0.0885%
